<a href="https://colab.research.google.com/github/Musadiq8699/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Musadiq8699/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Supervised Learning — Binary Classification with Probability Scoring

**Classification:** We categorize each unique page (content_hash_id) into two classes: 1 (Needs Content Refresh / At Risk of Severe Decay) or 0 (Healthy / Stable).

**Scoring:** The classification algorithm outputs a continuous confidence probability (e.g., 0.87 or 87% risk). We use this probability as a priority score to rank-order pages, allowing editorial teams to allocate limited rewriting resources to the highest-risk pages first.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task Type: Supervised Classification & Scoring")
print("Target Classes: 0 (Healthy) vs 1 (Needs Refresh)")


Task Type: Supervised Classification & Scoring
Target Classes: 0 (Healthy) vs 1 (Needs Refresh)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**What I am predicting:**
Whether a specific web page will suffer a significant traffic drop and needs to be updated (1 = Needs Refresh, 0 = Healthy).

**Where the label comes from (Proxy Target):**
Since "content decay" isn't a pre-existing column in the raw data, I create a proxy target label by comparing two time periods in Search Console data:Calculate the percentage change in clicks:$$\text{Click Delta} = \frac{\text{Recent Clicks} - \text{Baseline Clicks}}{\text{Baseline Clicks}}$$Set the ground-truth target label (is_decayed):1 (Needs Refresh): If a page lost $20\%$ or more of its clicks ($\text{Click Delta} \le -0.20$).0 (Healthy): If the page traffic remained stable or grew ($\text{Click Delta} > -0.20$).

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd


demo_df = pd.DataFrame({
    'page_id': ['Page A', 'Page B', 'Page C'],
    'baseline_clicks': [1000, 500, 1200],
    'recent_clicks': [700, 480, 1300]
})

demo_df['click_change'] = (demo_df['recent_clicks'] - demo_df['baseline_clicks']) / demo_df['baseline_clicks']
demo_df['is_decayed'] = (demo_df['click_change'] <= -0.20).astype(int)

demo_df[['page_id', 'click_change', 'is_decayed']]

,page_id,click_change,is_decayed
0,Page A,-0.300000,1
1,Page B,-0.040000,0
2,Page C,0.083333,0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Chosen Metric:** Precision (specifically Precision@K for the top flagged pages)

**Why this metric:**

Rewriting content requires valuable human writer time. If the model makes false positive errors (flagging healthy pages by mistake), content teams will waste hours updating articles that didn't need work. High Precision guarantees that when the model flags a page for a refresh, it actually needs one.

**What number means 'good'?**
Precision $\ge 0.80$ (80%): At least 8 out of every 10 pages flagged by the model must be genuine traffic decay cases.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.metrics import precision_score

y_actual = [1, 0, 1, 1, 0, 1, 0]
y_predicted = [1, 0, 1, 1, 0, 0, 0]

model_precision = precision_score(y_actual, y_predicted)
print(f"Target Metric: Precision")
print(f"Calculated Precision: {model_precision:.0%}")




Target Metric: Precision
Calculated Precision: 100%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:**
One row = One unique web page (content_hash_id)

**Explanation:**

To evaluate content decay, data must be aggregated at the individual content piece level. Each row in our DataFrame represents a distinct anonymized URL/page along with its aggregated historical metrics (impressions, clicks, average position, CTR) and calculated features over the observation window.

This ensures our ML model makes predictions per page, producing a clear, actionable list of URLs for editorial review.

In [22]:
# Clone your specific repository
!git clone https://github.com/Musadiq8699/flyrank-ml-internship.git

# Move into the cloned directory
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 120, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 120 (delta 35), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (120/120), 1.85 MiB | 11.19 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd

df = pd.read_csv('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')

page_col = 'content_id'


print(f"Total rows (unique pieces of content): {len(df)}")
print(f"Is '{page_col}' unique per row? {df[page_col].is_unique}")


df[[page_col, 'clicks_90d', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days']].head()


Total rows (unique pieces of content): 30000
Is 'content_id' unique per row? True


,content_id,clicks_90d,impressions_90d,ctr,avg_position,content_age_days
0,content_304f48230142,29,3803,0.76,10.6,187
1,content_a1fb4e703a9e,7,15320,0.05,20.3,445
2,content_9aa793d4d895,11,12581,0.09,36.5,141
3,content_331d6c4de07b,58,11751,0.49,6.2,463
4,content_d99b7a2d90ca,24,19140,0.13,44.0,263


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why simple rules fail:**

**Multi-Variable Interactions:**

Decay is rarely driven by a single metric. A page might maintain high impression volume while suffering subtle rank slips and CTR erosion. A static if/else rule (e.g., if traffic_drop > 20%) cannot dynamically weigh complex, non-linear interactions across 5+ metrics simultaneously.

**False Positives from Noise: **

Static rules trigger false alarms on temporary seasonal dips, external search engine updates, or minor fluctuation noise, leading writers to waste hours editing healthy articles.

**Reactive vs. Predictive: **

Fixed rules only trigger after severe traffic loss has already occurred. Machine learning learns subtle historical leading indicators to predict risk proactively, allowing teams to save rankings before traffic collapses.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

rule_based_flagged = df[df['ctr'] < 0.02].shape[0]
print(f"Pages flagged by simple static CTR rule (<2%): {rule_based_flagged}")
print(f"Total dataset pages evaluated: {len(df)}")
print("ML evaluates joint distribution across position, impressions, and CTR decay together.")

Pages flagged by simple static CTR rule (<2%): 13290
Total dataset pages evaluated: 30000
ML evaluates joint distribution across position, impressions, and CTR decay together.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.